<div style="font-family: 'Segoe UI', Arial, sans-serif; 
            border-left: 5px solid #1f77b4; 
            padding: 20px; 
            background-color: #f8f9fa; 
            border-radius: 4px; 
            box-shadow: 0 1px 3px rgba(0,0,0,0.05); 
            max-width: 800px; 
            margin-bottom: 25px;">
    
<h1 style="color: #212529; margin-top: 0; margin-bottom: 5px; font-size: 26px; font-weight: 700;">
    Part I: Simplified CMS Open Data
</h1>
<br>
<p style="color: #6c757d; margin-top: 0; margin-bottom: 20px; font-size: 18px; font-weight: bold;">
    Notebook 1: Dataset Overview
</p>

<hr style="border: 0; border-top: 1px solid #dee2e6; margin-bottom: 15px;">

<table style="border-collapse: collapse; width: 100%; font-size: 14px; line-height: 1.6;">
    <tr>
        <td style="padding: 4px 0; width: 150px; font-weight: bold; color: #495057;">Author:</td>
        <td style="padding: 4px 0; color: #212529;">Uditangshu Roy</td>
    </tr>
    <tr>
        <td style="padding: 4px 0; font-weight: bold; color: #495057;">Affiliation:</td>
        <td style="padding: 4px 0; color: #212529;">Undergraduate, Department of Physics and Astronomy, The University of Manchester</td>
    </tr>
    <tr>
        <td style="padding: 4px 0; font-weight: bold; color: #495057;">Contact:</td>
        <td style="padding: 4px 0; color: #1f77b4; text-decoration: none;">roy.uditangshu@gmail.com</td>
    </tr>
    <tr>
        <td style="padding: 4px 0; font-weight: bold; color: #495057;">Date:</td>
        <td style="padding: 4px 0; color: #212529;">August 2026</td>
    </tr>
    <tr>
        <td style="padding: 4px 0; font-weight: bold; color: #495057;">Dataset Source:</td>
        <td style="padding: 4px 0; color: #212529;">CERN Open Data: <i> Z to two muons from 2011</i>. McCauley, T. Recorded 2011. Published 2019.</td>
    </tr>
    <tr>
        <td style="padding: 4px 0; font-weight: bold; color: #495057;">Software Env.:</td>
        <td style="padding: 4px 0; color: #212529;">Python 3.11+, Pandas, Matplotlib, NumPy, Pathlib, IPython</td>
    </tr>
    <tr>
        <td style="padding: 4px 0; font-weight: bold; color: #495057;">ORCID:</td>
        <td style="padding: 4px 0; color: #212529; font-family: monospace; letter-spacing: 0.5px;"><a href="https://orcid.org/0009-0005-7992-3549">0009-0005-7992-3549</a></td>
    </tr>
</table>
</div>

## 1. &nbsp; Introduction
This notebook introduces the simplified CMS Open Data used throughout Part I of this project. The aim is to understand the structure of the dataset, inspect the available variables, verify the integrity of the imported data, and gain a preliminary understanding of the physical quantities recorded for each event before performing any kinematic reconstruction.

In [1]:
#Import libraries and modules

import sys
from pathlib import Path
from IPython.display import display, HTML
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utilities import (
    KINEMATIC_COLUMNS,
    TRACK_COLUMNS,
    TABLE_STYLES,
    make_formatter,
    dataset_summary,
    dataset_information,
    dataset_dimensions,
    event_statistics,
    missing_values
)

#Center-align outputs
display(HTML("""
<style>
.jp-OutputArea-output {
    display: flex;
    justify-content: center;
    align-items: center;
}
</style>
"""))

## 2. &nbsp; Project Information

### 2.1 &nbsp; Dataset Origin
The data analysed in this notebook are obtained from the CMS Open Data programme, an initiative that provides public access to data recorded by the Compact Muon Solenoid (CMS) experiment at the Large Hadron Collider (LHC). The dataset used in this first part of the project is a simplified educational sample containing approximately 10,000 pre-selected $Z \rightarrow \mu^+\mu^-$ candidate events. For each event, the reconstructed kinematic properties of the two muons, including their transverse momentum $(p_T)$, pseudorapidity $(\eta)$, azimuthal angle $(\phi)$, electric charge, and detector-related quantities (such as run and event numbers), are provided in a compact CSV format. This curated dataset is intended for exploring the fundamental principles of particle reconstruction without the complexity of a full experimental analysis.

### 2.2 &nbsp; Dataset Semantics
The dataset is of the following structure:

<div style="font-family: 'Segoe UI', Arial, sans-serif; 
            margin: 15px auto 25px auto; 
            max-width: 800px;
            border-left: 5px solid #165683; 
            padding: 8px; 
            background-color: #f8f9fa; 
            border-radius: 4px; 
            box-shadow: 0 1px 3px rgba(0,0,0,0.05);
            text-align:center;">
    <h3 style="color: #333; margin-bottom: 12px; font-size:15px; text-align:left">Table 1: Dataset Variable Dictionary</h3>
    <table style="border-collapse: collapse; width: 100%; box-shadow: 0 2px 5px rgba(0,0,0,0.05); border-radius: 4px; overflow: hidden; text-align:left;">
        <thead>
            <tr style="background-color: #343a40; color: white; text-align: left;">
                <th style="padding: 12px; font-size: 14px; width: 120px;">Variable</th>
                <th style="padding: 12px; font-size: 14px; width: 100px;">Unit / Type</th>
                <th style="padding: 12px; font-size: 14px;">Description</th>
            </tr>
        </thead>
        <tbody>
            <tr style="background-color: #f8f9fa; border-bottom: 1px solid #dee2e6;">
                <td style="padding: 10px 12px; font-family: monospace; font-weight: bold; color: #1f77b4;">Run</td>
                <td style="padding: 10px 12px; color: #6c757d; font-size: 13px;">Integer</td>
                <td style="padding: 10px 12px; color: #495057;">The run number of the event.</td>
            </tr>
            <tr style="background-color: #ffffff; border-bottom: 1px solid #dee2e6;">
                <td style="padding: 10px 12px; font-family: monospace; font-weight: bold; color: #1f77b4;">Event</td>
                <td style="padding: 10px 12px; color: #6c757d; font-size: 13px;">Integer</td>
                <td style="padding: 10px 12px; color: #495057;">The event number.</td>
            </tr>
            <tr style="background-color: #f8f9fa; border-bottom: 1px solid #dee2e6;">
                <td style="padding: 10px 12px; font-family: monospace; font-weight: bold; color: #1f77b4;">pt</td>
                <td style="padding: 10px 12px; color: #e83e8c; font-weight: 500; font-size: 13px;">GeV</td>
                <td style="padding: 10px 12px; color: #495057;">The transverse momentum ($p_T$) of the muon.</td>
            </tr>
            <tr style="background-color: #ffffff; border-bottom: 1px solid #dee2e6;">
                <td style="padding: 10px 12px; font-family: monospace; font-weight: bold; color: #1f77b4;">eta</td>
                <td style="padding: 10px 12px; color: #6c757d; font-size: 13px;">Float</td>
                <td style="padding: 10px 12px; color: #495057;">The pseudorapidity ($\eta$) of the muon.</td>
            </tr>
            <tr style="background-color: #f8f9fa; border-bottom: 1px solid #dee2e6;">
                <td style="padding: 10px 12px; font-family: monospace; font-weight: bold; color: #1f77b4;">phi</td>
                <td style="padding: 10px 12px; color: #e83e8c; font-weight: 500; font-size: 13px;">rad</td>
                <td style="padding: 10px 12px; color: #495057;">The azimuthal angle ($\phi$) of the muon.</td>
            </tr>
            <tr style="background-color: #ffffff; border-bottom: 1px solid #dee2e6;">
                <td style="padding: 10px 12px; font-family: monospace; font-weight: bold; color: #1f77b4;">Q</td>
                <td style="padding: 10px 12px; color: #6c757d; font-size: 13px;">&plusmn;1</td>
                <td style="padding: 10px 12px; color: #495057;">The electric charge of the muon.</td>
            </tr>
            <tr style="background-color: #f8f9fa; border-bottom: 1px solid #dee2e6;">
                <td style="padding: 10px 12px; font-family: monospace; font-weight: bold; color: #1f77b4;">dxy</td>
                <td style="padding: 10px 12px; color: #e83e8c; font-weight: 500; font-size: 13px;">cm</td>
                <td style="padding: 10px 12px; color: #495057;">The impact parameter in the transverse plane calculated with respect to the primary vertex.</td>
            </tr>
            <tr style="background-color: #ffffff;">
                <td style="padding: 10px 12px; font-family: monospace; font-weight: bold; color: #1f77b4;">iso</td>
                <td style="padding: 10px 12px; color: #6c757d; font-size: 13px;">Scalar (Float)</td>
                <td style="padding: 10px 12px; color: #495057;">The combined relative isolation ($I_{\text{track}} + I_{\text{ecal}} + I_{\text{hcal}}$) of the muon.</td>
            </tr>
        </tbody>
    </table>
</div>

The above sturcture is presented twice for the two muons, with the suffix of 1 or 2 denoting the first or the second muon's properties respectively.

#### Definitions
- **Event**:  An event is a collision of particles occuring in the detector. In LHC, it is primarily a collision of protons.
- **Muon**: An elementary particle belonging to the second generation of leptons, with an electric charge of -1. It has properties similar to those of an electron, but it is about 200 times more masssive.
- **Anti-Muon**: The anti-particle of Muon with an electric charge of +1.
- **Pseudorapidity**: The pseudorapidity $(\eta)$ is a coordinate that describes the angle of a particle produced in an event relative to the beam axis. $\eta = 0$ denotes that the produced particle is perpendicular to the beam axis.
- **Azimuthal Angle**: The azimuthal angle $(\phi)$ is the angle measured around the beam pipe (commonly the $z$-axis) in the transverse plane. In this case it ranges from $- \pi$ to $\pi$ angle.
- **SUB-DETECTORS**:
  - **Tracker**: The tracker is the innermost layer that maps the trajectory of charged particles and measures their momentum.
  - **ECAL**: The Electromagnetic Calorimeter measures the energies of electrons and photons by stopping them.
  - **HCAL**: The Hadron Calorimeter is the outermost layer, designed to stop hadrons (particles made of quarks and gluons, like protons and neutrons) and measure their energies.
  

### 2.3 &nbsp; Data Selection Procedure
For the double muon $Z \rightarrow \mu^+\mu^-$ dataset, an event was selected if there were two muons in the event with $p_T > 20 \: \text{GeV}$ and $|\eta| < 2.1$ and the invariant mass of the two muons was in the range of 60 GeV to 120 GeV.

### 2.4 &nbsp; Project Objectives
The primary objective of this notebook is to inspect the structure and contents of the dataset before undertaking any physics calculations. By verifying the integrity of the imported data, understanding the meaning of each recorded variable, and examining the overall characteristics of the sample, a solid foundation is established for the subsequent reconstruction of the $Z$-boson invariant mass and the more advanced analyses presented later in this project.

## 3. &nbsp; Load Dataset and Description

In [2]:
zmumu = pd.read_csv("../../data/raw/simplified/zmumu_10k.csv")

### 3.1 &nbsp; Initial Dataset Inspection

In [3]:
(zmumu.head()
    .rename(index=lambda x: x + 1)
    .style.set_caption("<b>Table 2:</b> First five events of the dimuon dataset.")
    .set_table_styles(TABLE_STYLES)
)

,Run,Event,pt1,eta1,phi1,Q1,dxy1,iso1,pt2,eta2,phi2,Q2,dxy2,iso2
1,165617,74969122,54.705500,-0.432400,2.574200,1,-0.074500,0.499900,34.246400,-0.988500,-0.498700,-1,0.071200,3.422100
2,165617,75138253,24.587200,-2.052200,2.866600,-1,-0.055400,0.000000,28.538900,0.385200,-1.991200,1,0.051500,0.000000
3,165617,75887636,31.738600,-2.259500,-1.332300,-1,0.087900,0.000000,30.234400,-0.468400,1.883300,1,-0.087600,0.000000
4,165617,75779415,39.739400,-0.712300,-0.312300,1,0.058500,0.000000,48.279000,-0.195600,2.970300,-1,-0.049200,0.000000
5,165617,75098104,41.299800,-0.157100,-3.040800,1,-0.030500,1.228000,43.450800,0.591000,-0.042800,-1,0.044200,0.000000


The first five entries of the dataset are displayed above as an example. Each row corresponds to a single proton–proton collision event in which a candidate $Z \rightarrow \mu^+\mu^-$ decay has been identified. The columns contain the reconstructed kinematic properties of the two muons produced in the decay, together with event identifiers that uniquely label the collision from which the candidate was reconstructed.

In [4]:
(dataset_information(zmumu)
    .style.set_caption("<b>Table 3:</b> Data Types")
    .set_table_styles(TABLE_STYLES)
)

,Data Type,Non-Null Entries
Run,int64,10000
Event,int64,10000
pt1,float64,10000
eta1,float64,10000
phi1,float64,10000
Q1,int64,10000
dxy1,float64,10000
iso1,float64,10000
pt2,float64,10000
eta2,float64,10000


The dataset information confirms that all variables have been imported successfully with appropriate numerical data types. The majority of the columns are stored as floating-point values, reflecting the continuous nature of the measured kinematic quantities, while the run number, event number, and particle charges are represented as integers. The absence of unexpected data types indicates that the dataset has been read correctly and is suitable for numerical analysis.

### 3.2 &nbsp; Dataset Descriptive Statistics

In [5]:
(dataset_summary(
    zmumu,
    KINEMATIC_COLUMNS + TRACK_COLUMNS
)
    .style.set_caption("<b>Table 4:</b> Dataset Descriptions")
    .set_table_styles(TABLE_STYLES)
)

,Mean,Standard Deviation,Minimum,Maximum
pt1,38.403000,14.428000,3.464000,269.080000
pt2,38.639000,15.945000,3.266000,528.434000
eta1,-0.279000,1.353000,-2.438000,2.100000
eta2,0.080000,0.872000,-2.428000,2.099000
phi1,-0.234000,1.805000,-3.141000,3.141000
phi2,0.251000,1.787000,-3.142000,3.141000
Q1,-0.031000,1.000000,-1.000000,1.000000
Q2,0.038000,0.999000,-1.000000,1.000000
dxy1,0.006000,0.075000,-3.588000,2.028000
dxy2,0.011000,1.781000,-2.005000,177.931000


The descriptive statistics provide an overview of the reconstructed kinematic properties of the selected $Z \rightarrow \mu^+\mu^-$ candidate events.

The transverse momentum $(p_T)$ of both muons has a mean value of approximately $38 \: \text{GeV}$, with a broad spread extending from around $3 \: \text{GeV}$ to over $500 \: \text{GeV}$. This wide range reflects the variety of event kinematics present within the sample while remaining consistent with the expectation that the decay products of a $91.2 \: \text{GeV} \: Z$-boson typically possess transverse momenta of several tens of GeV.

The pseudorapidity $(\eta)$ values lie approximately within the interval $-2.4 < \eta < 2.4$, corresponding to the acceptance of the CMS detector for reconstructed muons. The distributions are centred close to zero, indicating that the selected events are approximately symmetric about the detector's central region, as expected for proton–proton collisions.

The azimuthal angle $(\phi)$ spans the full range from approximately $-\pi$ to $\pi$, demonstrating that no preferred direction exists in the plane transverse to the beam axis. Likewise, the charge variables $Q_1$ and $Q_2$ contain only the values $\pm1$, confirming that the reconstructed particles are identified as charged muons.

Finally, the impact parameter $(d_{xy})$ is centred close to zero for both muons, indicating that the reconstructed tracks originate near the primary interaction vertex. Although the isolation variables exhibit relatively large maximum values, their median values are zero, suggesting that the majority of selected muons are well isolated, while a small number of events contain significantly larger isolation values.

### 3.3 &nbsp; Singular Event Description

The data structure is further preliminarily analysed by looking at a singular event, i.e., one row corresponding to a single proton-proton collision.

In [6]:
(zmumu.iloc[0]
    .map(make_formatter()).to_frame(name="First Event Values")
    .style.set_caption("<b>Table 5:</b> Singular row")
    .set_table_styles(TABLE_STYLES)
)

,First Event Values
Run,165617
Event,74969122
pt1,54.7055
eta1,-0.4324
phi1,2.5742
Q1,1
dxy1,-0.0745
iso1,0.4999
pt2,34.2464
eta2,-0.9885


The table above displays the recorded information for the first event in the dataset. Each row of the dataset corresponds to a single proton–proton collision in which a candidate $Z \rightarrow \mu^+\mu^-$ decay has been reconstructed. The Run and Event identifiers uniquely label the collision from which the candidate originates, while the remaining variables describe the reconstructed properties of the two detected muons.

For this event, the first muon has a transverse momentum of approximately $54.7 \: \text{GeV}$, while the second has a transverse momentum of approximately $34.2 \, \text{GeV}$. Both particles are reconstructed within the fiducial acceptance of the CMS detector, as indicated by their pseudorapidity values $(|\eta| < 2.4)$. The azimuthal angles show that the muons emerge in different directions within the transverse plane, as expected for particles produced in a high-energy collision.

The reconstructed charges are $+1$ and $-1$, demonstrating that the two particles form an oppositely charged muon pair, consistent with the expected signature of a $Z$-boson decay. The transverse impact parameters $(d_{xy})$ are both small and centred close to zero, indicating that the tracks originate close to the primary interaction vertex. Finally, while the first muon is relatively well isolated, the second exhibits a larger isolation value, illustrating that individual events can display varying levels of surrounding detector activity even within this dataset.

## 4. &nbsp; Data Integrity Checks

### 4.1 &nbsp; Dataset Dimensions
The dimensions of the dataset are first inspected to verify that the expected number of events and variables have been imported successfully.

In [7]:
dataset_dimensions(zmumu)

Rows    : 10,000
Columns : 14


(10000, 14)

The dataset contains 10,000 recorded events, each described by 14 variables. This agrees with the expected size of the simplified CMS Open Data sample and confirms that the complete dataset has been imported successfully and in its entirety.

### 4.2 &nbsp; Missing Data Values
The presence of missing values is examined to ensure that every event contains a complete set of measurements before any physics calculations are performed.

In [8]:
(missing_values(zmumu)
    .style.set_caption("<b>Table 6:</b> Missing values")
    .set_table_styles(TABLE_STYLES)
)

,Missing Values,Percentage (%)
Run,0,0.00
Event,0,0.00
pt1,0,0.00
eta1,0,0.00
phi1,0,0.00
Q1,0,0.00
dxy1,0,0.00
iso1,0,0.00
pt2,0,0.00
eta2,0,0.00


No missing values are present in any of the recorded variables. This indicates that every event contains a complete set of measurements required for the subsequent reconstruction of the invariant mass, and no additional preprocessing or imputation of missing data is necessary.

### 4.3 &nbsp; Duplicate Events
Similar to section 4.2 on missing data values, this sub-section examines the presence of duplicate events to ensure that every entry refers to a unique recorded event.

In [9]:
print(f"Duplicate rows: {zmumu.duplicated().sum()}")

Duplicate rows: 0


No duplicate rows were identified within the dataset. Each entry therefore corresponds to a unique recorded event, eliminating the possibility of double-counting during the subsequent statistical analysis.

### 4.4 &nbsp; Event Identifiers

In [10]:
event_statistics(zmumu)

Total Events   : 10,000
Variables      : 14
Unique Runs    : 19
Unique Events  : 10000


The number of unique event identifiers is equal to the total number of recorded events, confirming that each collision event appears only once within the dataset.
The dataset contains events originating from multiple data-taking runs. The run number identifies the specific period during which the detector recorded the collision, while the event number uniquely labels individual collisions within each run. These identifiers are primarily used for event bookkeeping and are not directly involved in the kinematic reconstruction performed later in this project.

## 5. &nbsp; Conclusion

The simplified CMS Open Data have been successfully imported and inspected. Basic validation confirms that the dataset contains no missing values or duplicate events and consists of 10,000 reconstructed $Z \rightarrow \mu^+\mu^-$ candidate events with complete kinematic information. The recorded variables and their statistical properties are consistent with expectations for high-energy proton–proton collisions. Having verified the integrity and structure of the dataset, the analysis can now proceed to a detailed exploration of the kinematic distributions before reconstructing the invariant mass of the parent $Z$ boson.